In [1]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel

torch.cuda.empty_cache()

# model_name = "meta-llama/Llama-3.2-1B"
model_name = "Qwen/Qwen3-0.6B"
model_save_dir = "model/downloaded"
adapter_save_dir = "model/downloaded/qwen3_0.6b_med_3eps_test"
os.makedirs(model_save_dir, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="cuda",
    cache_dir=model_save_dir
)
peft_model = PeftModel.from_pretrained(
    model,
    adapter_save_dir)

In [2]:
from pruning.prune import foresight_prune, prune_wanda
from data.datasets import get_loaders

dataloader, _ = get_loaders(
    name="harrison",
    tokenizer=tokenizer,
    seqlen=2048,
    nsamples=128)

foresight_prune(model=peft_model,
                dataloader=dataloader,
                prune_ratio=0.6,
                mask_lr=0.5,
                nsamples=128)

# model = peft_model.merge_and_unload()
# prune_wanda(sparsity_ratio=0.4,
#             nsamples=128,
#             seed=42,
#             seqlen=2048,
#             model=model,
#             tokenizer=tokenizer,
#             dataset_name="harrison")

Token indices sequence length is longer than the specified maximum sequence length for this model (4914897 > 131072). Running this sequence through the model will result in indexing errors


preparing calibration input
calibration input prepared
Pruning layer 0, param self_attn.q_proj
Initializing score tensor
This weight is not being recognized self_attn.q_proj
Performing PBS recovery
Pruning layer 0, param self_attn.k_proj
Initializing score tensor
Performing PBS recovery
Pruning layer 0, param self_attn.v_proj
Initializing score tensor
Performing PBS recovery
Pruning layer 0, param self_attn.o_proj
Initializing score tensor
Performing PBS recovery
Pruning layer 0, param mlp.gate_proj
Initializing score tensor
Performing PBS recovery
Pruning layer 0, param mlp.up_proj
Initializing score tensor
Performing PBS recovery
Pruning layer 0, param mlp.down_proj
Initializing score tensor
Performing PBS recovery
Pruning layer 1, param self_attn.q_proj
Initializing score tensor
This weight is not being recognized self_attn.q_proj
Performing PBS recovery
Pruning layer 1, param self_attn.k_proj
Initializing score tensor
Performing PBS recovery
Pruning layer 1, param self_attn.v_proj


In [ ]:
from pruning.utils import check_sparsity
model = peft_model.merge_and_unload()
sparsity = check_sparsity(model)
print(f"Sparsity after pruning: {sparsity:.2%}")
model.save_pretrained("model/downloaded/qwen3_0.6b_med_3eps_test_foresightpruned")

layer 0 sparsity 0.599674
layer 1 sparsity 0.599675
layer 2 sparsity 0.599674
layer 3 sparsity 0.599674
layer 4 sparsity 0.599675
layer 5 sparsity 0.599675
layer 6 sparsity 0.599675
layer 7 sparsity 0.599675
layer 8 sparsity 0.599674
layer 9 sparsity 0.599674
layer 10 sparsity 0.599675
layer 11 sparsity 0.599675
layer 12 sparsity 0.599674
layer 13 sparsity 0.599675
layer 14 sparsity 0.599675
layer 15 sparsity 0.599675
layer 16 sparsity 0.599675
layer 17 sparsity 0.599674
layer 18 sparsity 0.599675
layer 19 sparsity 0.599675
layer 20 sparsity 0.599675
layer 21 sparsity 0.599674
layer 22 sparsity 0.599675
layer 23 sparsity 0.599674
layer 24 sparsity 0.599675
layer 25 sparsity 0.599675
layer 26 sparsity 0.599675
layer 27 sparsity 0.599674
Sparsity after pruning: 59.97%


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [6]:
from evaluation.perplexity import eval_ppl
ppl = eval_ppl(model = model, tokenizer=tokenizer, dataset="harrison", seqlen=2048)
print (f"Perplexity after pruning: {ppl}")

evaluating on harrison


block 0/300
block 50/300
block 100/300
block 150/300
block 200/300
block 250/300
Perplexity after pruning: 22.045287138581553


In [4]:
import torch

@torch.inference_mode()
def prefill_throughput_tokens_per_s(model, tokenizer, seqlen=2048, batch_size=1, niter=100, warmup=10):
    model.eval()

    device = next(model.parameters()).device
    if device.type != "cuda":
        raise ValueError("Model must be on CUDA for this benchmark.")

    input_ids = torch.randint(
        low=0,
        high=tokenizer.vocab_size,
        size=(batch_size, seqlen),
        dtype=torch.long,
        device=device,
    )
    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=device)

    # Warm-up
    for _ in range(warmup):
        _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()
    for _ in range(niter):
        _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
    end.record()

    torch.cuda.synchronize()
    ms = start.elapsed_time(end)
    seconds = ms / 1000.0

    tokens = batch_size * seqlen * niter
    return tokens / seconds

throughput = prefill_throughput_tokens_per_s(model, tokenizer, seqlen=2048, batch_size=1, niter=50, warmup=10)
print(f"Prefill throughput: {throughput:.2f} tokens/s")

Prefill throughput: 57932.29 tokens/s


In [4]:
import torch
import numpy as np

def _percentile(x, q):
    return float(np.percentile(np.asarray(x, dtype=np.float64), q))

@torch.inference_mode()
def measure_latency_ms(
    model,
    tokenizer=None,
    *,
    mode: str = "prefill",          # "prefill" or "generate"
    prompt_text: str | None = None, # if provided, uses real tokenization
    seqlen: int = 2048,             # used when prompt_text is None (synthetic)
    batch_size: int = 1,
    max_new_tokens: int = 256,      # used for mode="generate"
    niter: int = 50,
    warmup: int = 10,
    use_cache: bool | None = None,  # None => model default
    do_sample: bool = False,
):
    """
    Returns a dict with latency statistics in milliseconds.

    Notes:
      - Requires model to be on CUDA.
      - Uses CUDA Events for accurate GPU timing.
      - For mode="generate", reports end-to-end latency and time-per-output-token (TPOT).
    """
    model.eval()
    device = next(model.parameters()).device
    if device.type != "cuda":
        raise ValueError("This latency benchmark expects the model to be on CUDA.")

    # Build inputs
    if prompt_text is not None:
        if tokenizer is None:
            raise ValueError("tokenizer is required when prompt_text is provided.")
        enc = tokenizer(
            [prompt_text] * batch_size,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc.get("attention_mask", None)
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)
    else:
        if tokenizer is None:
            raise ValueError("tokenizer is required for synthetic inputs (vocab_size needed).")
        input_ids = torch.randint(
            low=0,
            high=tokenizer.vocab_size,
            size=(batch_size, seqlen),
            dtype=torch.long,
            device=device,
        )
        attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=device)

    # Warm-up
    for _ in range(warmup):
        if mode == "prefill":
            _ = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=use_cache,
            )
        elif mode == "generate":
            _ = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                use_cache=True if use_cache is None else use_cache,
            )
        else:
            raise ValueError('mode must be "prefill" or "generate".')

    torch.cuda.synchronize()

    # Timed iterations (per-iteration latency distribution)
    times_ms = []
    for _ in range(niter):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)

        start.record()
        if mode == "prefill":
            _ = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=use_cache,
            )
        else:  # generate
            _ = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                use_cache=True if use_cache is None else use_cache,
            )
        end.record()

        torch.cuda.synchronize()
        times_ms.append(start.elapsed_time(end))

    # Stats
    stats = {
        "mode": mode,
        "niter": niter,
        "batch_size": batch_size,
        "input_len": int(input_ids.shape[1]),
        "mean_ms": float(np.mean(times_ms)),
        "p50_ms": _percentile(times_ms, 50),
        "p90_ms": _percentile(times_ms, 90),
        "p95_ms": _percentile(times_ms, 95),
        "p99_ms": _percentile(times_ms, 99),
    }

    # Optional derived metrics
    total_s = (np.sum(times_ms) / 1000.0)
    if mode == "prefill":
        total_tokens = batch_size * int(input_ids.shape[1]) * niter
        stats["tokens_per_s"] = float(total_tokens / total_s)
    else:
        # End-to-end generate latency; TPOT uses requested new tokens (fixed here).
        gen_tokens = batch_size * max_new_tokens
        stats["tpot_ms_per_token"] = float(stats["mean_ms"] / gen_tokens)
        stats["gen_tokens_per_s"] = float((gen_tokens * niter) / total_s)

    return stats


latency_stats = measure_latency_ms(
    model,
    tokenizer=tokenizer,
    mode="prefill",
    seqlen=2048,
    batch_size=10,
    niter=100,
    warmup=10
)
print("Latency stats (prefill):")
for k, v in latency_stats.items():
    print(f"  {k}: {v}")

ValueError: This latency benchmark expects the model to be on CUDA.

In [5]:
import torch

def bytes_to_gb(x: int) -> float:
    return x / (1024**3)

@torch.inference_mode()
def measure_model_load_vram_gb(model, device="cuda"):
    """
    Measures GPU memory increase after moving `model` to GPU.
    Returns allocated/reserved deltas (GB) and parameter bytes (GB).
    """
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available.")

    torch.cuda.synchronize()
    torch.cuda.empty_cache()

    # Baseline
    base_alloc = torch.cuda.memory_allocated()
    base_resv  = torch.cuda.memory_reserved()

    # Move model
    model = model.to(device)
    torch.cuda.synchronize()

    # After
    after_alloc = torch.cuda.memory_allocated()
    after_resv  = torch.cuda.memory_reserved()

    # Deltas
    delta_alloc = after_alloc - base_alloc
    delta_resv  = after_resv  - base_resv

    # Parameter bytes (theoretical lower bound for weights)
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())

    return {
        "delta_allocated_gb": bytes_to_gb(delta_alloc),
        "delta_reserved_gb": bytes_to_gb(delta_resv),
        "param_bytes_gb": bytes_to_gb(param_bytes),
        "note": "delta_reserved includes allocator caching; delta_allocated is closer to actual live tensors."
    }

vram_stats = measure_model_load_vram_gb(model, device="cuda")
print("VRAM usage after loading model:")
for k, v in vram_stats.items():
    print(f"  {k}: {v}")

VRAM usage after loading model:
  delta_allocated_gb: 1.110243320465088
  delta_reserved_gb: 0.021484375
  param_bytes_gb: 1.1102294921875
  note: delta_reserved includes allocator caching; delta_allocated is closer to actual live tensors.


In [7]:
from evaluation.domain_zero_shot import evaluate_pubmedqa, evaluate_mednli, evaluate_hqs

post = {}
acc, maf, cm, predictions = evaluate_pubmedqa(model, tokenizer)
print(f"PubMedQA Accuracy after fine-tuning: {acc:.4f}, Macro F1: {maf:.4f}")
post["PubMedQA"] = {"accuracy": float(acc), "macro_f1": float(maf)}
acc, maf, cm, predictions = evaluate_mednli(model, tokenizer)
print(f"MedNLI Accuracy after fine-tuning: {acc:.4f}, Macro F1: {maf:.4f}")
post["MedNLI"] = {"accuracy": float(acc), "macro_f1": float(maf)}
rouge, prediction = evaluate_hqs(model, tokenizer)
print("HQS ROUGE after fine-tuning:", rouge)
post["HQS"] = {"rouge": rouge}

Predicting final decisions: 100%|██████████| 500/500 [01:22<00:00,  6.06it/s]


=== PubMedQA Results ===
Saved detailed log to: /home/918839576/Trepo/Efficient_Domain_Adaptation/assets/pubmedqa_predictions.txt
Accuracy = 0.6460
Macro-F1 = 0.4419
Macro-F1 95% bootstrap CI (n_boot=1000): [0.4111, 0.4702]
PubMedQA Accuracy after fine-tuning: 0.6460, Macro F1: 0.4419


Evaluating MedNLI: 100%|██████████| 1422/1422 [03:54<00:00,  6.06it/s]


=== MedNLI Results ===
Saved detailed log to: /home/918839576/Trepo/Efficient_Domain_Adaptation/assets/mednli_predictions.txt
Accuracy = 0.5373
Macro-F1 = 0.5206
Macro-F1 95% bootstrap CI (n_boot=1000): [0.4939, 0.5468]
MedNLI Accuracy after fine-tuning: 0.5373, Macro F1: 0.5206


Generating Summaries: 100%|██████████| 100/100 [01:21<00:00,  1.23it/s]


=== Final ROUGE SCORES ===
ROUGE-1: 0.3073
ROUGE-2: 0.1122
ROUGE-L: 0.2791

Total generation and evaluation time: 81.09 seconds
HQS ROUGE after fine-tuning: {'rouge1': np.float64(0.30733941998540704), 'rouge2': np.float64(0.11217187543857168), 'rougeL': np.float64(0.2790899088971906), 'rougeLsum': np.float64(0.2801419551934694)}
